# 🧩 Per-Question Imbalanced-Classification Model Trainer
### DHS Couples Data — predicting each survey answer from husband + wife demographics

**What this notebook does (read me first):**

* Trains **one independent model per survey question** (9 models total).
* For every question the rare **"red-flag / negative" answer** is the class we care about — the notebook is built to **catch those rare cases (high recall)** *without* drowning in false alarms (it does **not** simply relabel the majority as the minority).
* Handles the heavy class imbalance with three stacked techniques: **class weighting → (optional) resampling → decision-threshold tuning**.
* **Picks the best model automatically** for each question from a zoo of strong open-source classifiers, judged on **Matthews Correlation Coefficient (MCC)** — the metric that is only high when *both* classes are predicted well (i.e. "balanced").
* Saves every trained model + its tuned threshold to disk so you can reuse them.

**How to navigate:** every section has a `BOOKMARK n` banner in its first comment line. Sections **10.1 – 10.9** are the nine question-specific models — run them in any order; each is self-contained once sections 1–9 have run.

> ⚙️ The notebook auto-detects `xgboost`, `lightgbm` and `imbalanced-learn`. If they are installed they are added to the model zoo; if not, it runs on **pure scikit-learn** and still works.

## `BOOKMARK 1` — Imports & environment detection

In [ ]:
# ===== BOOKMARK 1 :: IMPORTS & ENVIRONMENT =====
import warnings, os, json, time, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, matthews_corrcoef,
                             f1_score, fbeta_score, recall_score, precision_score,
                             balanced_accuracy_score, average_precision_score, roc_auc_score)

# ---- Optional power-libraries (used automatically if present) ----
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except Exception:
    HAS_LGB = False
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    from imblearn.ensemble import BalancedRandomForestClassifier
    HAS_IMB = True
except Exception:
    HAS_IMB = False

print("xgboost         :", "available" if HAS_XGB else "NOT installed (skipped)")
print("lightgbm        :", "available" if HAS_LGB else "NOT installed (skipped)")
print("imbalanced-learn:", "available" if HAS_IMB else "NOT installed (skipped)")
print("\nEverything below runs with or without the optional libraries.")

## `BOOKMARK 2` — Configuration (edit knobs here)

Everything you might want to tweak lives in one place.

* **`THRESHOLD_OBJECTIVE`** controls the recall/precision balance of the final decision cut-off:
  * `"mcc"`  → most *balanced* (default; recommended).
  * `"f1"`   → leans toward minority precision/recall balance.
  * `"f2"`   → **recall-priority**: catch as many rare cases as possible (use if missing a true case is costly).
  * `"recall_floor"` → smallest threshold that still gives at least `RECALL_FLOOR` recall on the rare class.

In [ ]:
# ===== BOOKMARK 2 :: CONFIG =====
CONFIG = {
    # ---- data ----
    "data_path"        : "completeresponse_dhs.csv",   # <- change if your file lives elsewhere
    "output_dir"       : "trained_models",             # where models + metadata are written
    # ---- splitting ----
    "test_size"        : 0.20,   # untouched final hold-out
    "val_frac_of_train": 0.25,   # carved from TRAIN for model+threshold selection
    "random_state"     : 42,
    # ---- model selection ----
    "selection_metric" : "mcc",  # metric used to pick the winning model on validation
    # ---- decision threshold tuning ----
    "threshold_objective": "mcc",  # "mcc" | "f1" | "f2" | "recall_floor"
    "recall_floor"     : 0.70,     # only used when threshold_objective == "recall_floor"
    # ---- speed / depth ----
    "rf_estimators"    : 300,
    "hgb_max_iter"     : 350,
    "perm_importance_sample": 3000,  # cap rows for permutation importance (speed)
    "perm_importance_repeats": 5,
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
RS = CONFIG["random_state"]
print("Config loaded. Models will be saved to:", os.path.abspath(CONFIG["output_dir"]))

## `BOOKMARK 3` — Load data

In [ ]:
# ===== BOOKMARK 3 :: LOAD DATA =====
df = pd.read_csv(CONFIG["data_path"])
print("Rows, Cols:", df.shape)
df.head(3)

## `BOOKMARK 4` — Demographic features & **male–female combination** features

The raw inputs are the 13 husband/wife demographic columns. Because your goal is to study the
**effect of the male–female *combination***, we also engineer relational features that compare the
two partners (age gap, who is older, education/religion/ethnicity match). These interaction features
are exactly where "combination effects" show up.

In [ ]:
# ===== BOOKMARK 4 :: FEATURE ENGINEERING =====
RAW_FEATURES = [
    "Husband_age", "Husband's Education Level", "Husband Occupation_(grouped)",
    "Husband Ethnicity", "Husband Religion",
    "Wife's_current_age", "Wife's height(centimeters)", "Wife's Education Level",
    "Wife's_occupation_(grouped)", "Wife's Ethnicity", "Wife's Religion",
    "residence", "State",
]

def engineer_features(d):
    """Add husband<->wife relational ('combination') features."""
    d = d.copy()
    # --- Age heterogamy ---
    d["Age_Gap"]        = d["Husband_age"] - d["Wife's_current_age"]
    d["Abs_Age_Gap"]    = d["Age_Gap"].abs()
    d["High_Age_Gap"]   = (d["Abs_Age_Gap"] >= 7).astype(int)     # literature: >=7 yrs = stressor
    d["Wife_Older"]     = (d["Wife's_current_age"] > d["Husband_age"]).astype(int)
    # --- Status (mis)match between partners ---
    d["Edu_Match"]      = (d["Husband's Education Level"] == d["Wife's Education Level"]).astype(int)
    d["Religion_Match"] = (d["Husband Religion"] == d["Wife's Religion"]).astype(int)
    d["Ethnicity_Match"]= (d["Husband Ethnicity"] == d["Wife's Ethnicity"]).astype(int)
    return d

df_feat = engineer_features(df)

NUMERIC_COLS = ["Husband_age", "Wife's_current_age", "Wife's height(centimeters)",
                "Age_Gap", "Abs_Age_Gap"]
ENGINEERED_BINARY = ["High_Age_Gap", "Wife_Older", "Edu_Match", "Religion_Match", "Ethnicity_Match"]
CATEGORICAL_COLS = [c for c in RAW_FEATURES if c not in NUMERIC_COLS] + ENGINEERED_BINARY
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS

print(f"{len(FEATURE_COLS)} model features ({len(NUMERIC_COLS)} numeric + {len(CATEGORICAL_COLS)} categorical).")
print("Engineered combination features:", ["Age_Gap","Abs_Age_Gap"]+ENGINEERED_BINARY)

## `BOOKMARK 5` — Define the "rare / negative" class for each question

Each survey question is turned into a **binary target** where **`1` = the rare adverse ("red-flag")
answer** we want to detect, and **`0` = the benign answer**. The exact rule for every question is
documented in the table and lives in `TARGET_SPECS` so you can adjust it.

| Question | Class **1** (rare / negative) | Class **0** |
|---|---|---|
| Jealous if wife talks to men | `Yes` | No / Don't know |
| Insists on knowing where wife is | `Yes` | No / Don't know |
| Ever humiliated | anything except `Never` | `Never` |
| Ever insulted / made to feel bad | anything except `Never` | `Never` |
| Ever pushed / shook / thrown at | anything except `Never` | `Never` |
| Ever slapped | anything except `Never` | `Never` |
| Decides wife's health care | Husband alone / Someone else / Other | Joint / Wife alone |
| Decides large purchases | Husband alone / Someone else / Other | Joint / Wife alone |
| Wife afraid of husband | **`Most of the time afraid`** (severe) | Never / Sometimes |

> For the *fear* question the rarest, most severe outcome is "most of the time afraid", so that is the
> red flag. If you would rather flag *any* fear, swap in `bz_any_fear` in `TARGET_SPECS`.

In [ ]:
# ===== BOOKMARK 5 :: TARGET DEFINITIONS (binarizers) =====
def _low(s):
    return s.astype(str).str.strip().str.lower()

def bz_yes(s):        return (_low(s) == "yes").astype(int)
def bz_not_never(s):  return (~_low(s).str.startswith("never")).astype(int)
def bz_decision(s):
    v = _low(s)
    bad = v.str.contains("husband/partner alone") | v.str.contains("someone else") | (v == "other")
    return bad.astype(int)
def bz_most_fear(s):  return _low(s).str.contains("most of the time").astype(int)
def bz_any_fear(s):   return (~_low(s).str.startswith("never")).astype(int)  # alt for fear question

# question column  ->  (binarizer, short_name, meaning of class 1)
TARGET_SPECS = {
    "Husband_partner_jealous_if_Wife_talks_with_other_men":
        (bz_yes,       "Jealousy",         "Husband IS jealous"),
    "Husband_partner_insists_on_knowing_where_Wife_is":
        (bz_yes,       "Controlling",      "Husband insists on knowing whereabouts"),
    "Ever_been_humiliated_by_husband_partner":
        (bz_not_never, "Humiliated",       "Has been humiliated"),
    "Ever_been_insulted_or_made_to_feel_bad_by_husband_partner":
        (bz_not_never, "Insulted",         "Has been insulted"),
    "Ever_been_pushed,_shook_or_had_something_thrown_by_husband_partner":
        (bz_not_never, "Pushed-Shook",     "Has been pushed/shook/thrown at"),
    "Ever_been_slapped_by_husband_partner":
        (bz_not_never, "Slapped",          "Has been slapped"),
    "Person_who_usually_decides_on_Wife's_health_care":
        (bz_decision,  "HealthDecision",   "Wife lacks say in her health care"),
    "Person_who_usually_decides_on_large_household_purchases":
        (bz_decision,  "PurchaseDecision", "Wife lacks say in large purchases"),
    "Wife_afraid_of_husband_partner_most_of_the_time,_sometimes_or_never":
        (bz_most_fear, "Fear",             "Wife afraid most of the time"),
}

print("Imbalance per question (share of rare class 1):")
for col, (fn, name, meaning) in TARGET_SPECS.items():
    y = fn(df[col]); print(f"  {name:16s} pos-rate={y.mean():6.2%}   [{meaning}]")

## `BOOKMARK 6` — Preprocessing pipeline (leak-free)

In [ ]:
# ===== BOOKMARK 6 :: PREPROCESSOR =====
def build_preprocessor():
    """Median-impute + scale numerics; mode-impute + one-hot encode categoricals.
       Wrapped in a ColumnTransformer so it is fit only on training folds (no leakage)."""
    num_pipe = SkPipeline([("impute", SimpleImputer(strategy="median")),
                           ("scale",  StandardScaler())])
    cat_pipe = SkPipeline([("impute", SimpleImputer(strategy="most_frequent")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num_pipe, NUMERIC_COLS),
                              ("cat", cat_pipe, CATEGORICAL_COLS)])

## `BOOKMARK 7` — The model zoo (candidates compared for every question)

All candidates are **imbalance-aware**. Tree/boosting models use class weighting or `scale_pos_weight`;
logistic regression uses balanced weights. If `imbalanced-learn` is installed, two **SMOTE-resampling**
variants and a **Balanced Random Forest** are added too.

In [ ]:
# ===== BOOKMARK 7 :: MODEL ZOO =====
def build_model_zoo(pos_weight):
    """Return {name: full_pipeline}. pos_weight = (#neg / #pos) for cost-sensitive learners."""
    zoo = {}
    # -- always-available scikit-learn learners (class-weight based) --
    zoo["LogReg_balanced"] = SkPipeline([("pre", build_preprocessor()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RS))])
    zoo["RandomForest_balanced"] = SkPipeline([("pre", build_preprocessor()),
        ("clf", RandomForestClassifier(n_estimators=CONFIG["rf_estimators"],
                                       class_weight="balanced_subsample",
                                       n_jobs=-1, random_state=RS))])
    zoo["HistGradBoost_balanced"] = SkPipeline([("pre", build_preprocessor()),
        ("clf", HistGradientBoostingClassifier(class_weight="balanced",
                                               max_iter=CONFIG["hgb_max_iter"],
                                               learning_rate=0.05, random_state=RS))])
    # -- optional: XGBoost --
    if HAS_XGB:
        zoo["XGBoost"] = SkPipeline([("pre", build_preprocessor()),
            ("clf", XGBClassifier(n_estimators=400, learning_rate=0.04, max_depth=5,
                                  subsample=0.8, colsample_bytree=0.8,
                                  scale_pos_weight=pos_weight, eval_metric="logloss",
                                  n_jobs=-1, random_state=RS))])
    # -- optional: LightGBM --
    if HAS_LGB:
        zoo["LightGBM"] = SkPipeline([("pre", build_preprocessor()),
            ("clf", LGBMClassifier(n_estimators=500, learning_rate=0.04, num_leaves=31,
                                   subsample=0.8, colsample_bytree=0.8,
                                   class_weight="balanced", n_jobs=-1, random_state=RS, verbose=-1))])
    # -- optional: imbalanced-learn resampling + BalancedRF --
    if HAS_IMB:
        zoo["SMOTE+LogReg"] = ImbPipeline([("pre", build_preprocessor()),
            ("smote", SMOTE(random_state=RS)),
            ("clf", LogisticRegression(max_iter=2000, random_state=RS))])
        zoo["SMOTE+HistGB"] = ImbPipeline([("pre", build_preprocessor()),
            ("smote", SMOTE(random_state=RS)),
            ("clf", HistGradientBoostingClassifier(max_iter=CONFIG["hgb_max_iter"],
                                                   learning_rate=0.05, random_state=RS))])
        zoo["BalancedRandomForest"] = ImbPipeline([("pre", build_preprocessor()),
            ("clf", BalancedRandomForestClassifier(n_estimators=CONFIG["rf_estimators"],
                                                   n_jobs=-1, random_state=RS))])
    return zoo

## `BOOKMARK 8` — Metrics & decision-threshold tuning

With imbalanced data the default 0.5 cut-off is almost never optimal. We scan thresholds and pick the
one that best satisfies `CONFIG["threshold_objective"]`. **MCC** is the headline selection metric: it is
high only when the model gets *both* the rare and the common class right, which is precisely the
"find the rare cases without mislabelling the majority" balance you asked for.

In [ ]:
# ===== BOOKMARK 8 :: METRICS & THRESHOLD TUNING =====
def tune_threshold(y_true, proba, objective="mcc", recall_floor=0.70):
    """Find the probability cut-off that optimises the chosen objective."""
    grid = np.linspace(0.02, 0.98, 49)
    best_t, best_score = 0.5, -np.inf
    for t in grid:
        pred = (proba >= t).astype(int)
        if objective == "mcc":
            score = matthews_corrcoef(y_true, pred)
        elif objective == "f1":
            score = f1_score(y_true, pred, zero_division=0)
        elif objective == "f2":
            score = fbeta_score(y_true, pred, beta=2, zero_division=0)
        elif objective == "recall_floor":
            rec = recall_score(y_true, pred, zero_division=0)
            score = precision_score(y_true, pred, zero_division=0) if rec >= recall_floor else -1 + rec
        else:
            raise ValueError(objective)
        if score > best_score:
            best_score, best_t = score, t
    return float(best_t)

def score_at(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "MCC"        : round(matthews_corrcoef(y_true, pred), 4),
        "BalancedAcc": round(balanced_accuracy_score(y_true, pred), 4),
        "Recall_rare": round(recall_score(y_true, pred, zero_division=0), 4),
        "Prec_rare"  : round(precision_score(y_true, pred, zero_division=0), 4),
        "F1_rare"    : round(f1_score(y_true, pred, zero_division=0), 4),
        "PR_AUC"     : round(average_precision_score(y_true, proba), 4),
        "ROC_AUC"    : round(roc_auc_score(y_true, proba), 4),
    }

## `BOOKMARK 9` — Core trainer (`train_question`)

This single function does everything for one question:
1. Build the binary target and split **train / validation / untouched-test** (stratified).
2. Fit every model in the zoo on *train*, score on *validation* (with a tuned threshold).
3. **Select the winner** by validation MCC.
4. Refit the winner on **train + validation**, lock the tuned threshold, and report honest metrics on the **untouched test set** (confusion matrix + full classification report).
5. Compute **permutation feature importance** (which husband/wife combinations drive the prediction).
6. **Save** the fitted pipeline + threshold + metadata to `output_dir`.

In [ ]:
# ===== BOOKMARK 9 :: CORE TRAINER =====
def train_question(qcol, show_plots=True):
    fn, short, meaning = TARGET_SPECS[qcol]
    y_all = fn(df[qcol]).values
    X_all = df_feat[FEATURE_COLS].copy()

    print("="*78)
    print(f"QUESTION: {short}   (class 1 = {meaning})")
    print(f"   column: {qcol}")
    print(f"   rare-class share = {y_all.mean():.2%}   (n={int(y_all.sum())} of {len(y_all)})")
    print("="*78)

    # --- 1. splits: TRAIN / VAL / TEST (test stays untouched) ---
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=CONFIG["test_size"], random_state=RS, stratify=y_all)
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_tr, y_tr, test_size=CONFIG["val_frac_of_train"], random_state=RS, stratify=y_tr)

    pos_weight = (y_fit == 0).sum() / max((y_fit == 1).sum(), 1)
    zoo = build_model_zoo(pos_weight)

    # --- 2. fit each candidate, score on validation ---
    leaderboard = []
    for name, pipe in zoo.items():
        t0 = time.time()
        try:
            pipe.fit(X_fit, y_fit)
            p_val = pipe.predict_proba(X_val)[:, 1]
            thr   = tune_threshold(y_val, p_val, CONFIG["threshold_objective"], CONFIG["recall_floor"])
            m     = score_at(y_val, p_val, thr)
            leaderboard.append({"model": name, "val_thr": thr, **m, "secs": round(time.time()-t0,1)})
        except Exception as e:
            print(f"   [skip] {name}: {e}")

    sel_col = {"mcc":"MCC","f1":"F1_rare"}.get(CONFIG["selection_metric"], "MCC")
    lb = pd.DataFrame(leaderboard).sort_values(sel_col, ascending=False).reset_index(drop=True)
    print("\nValidation leaderboard (sorted by", sel_col, "):")
    print(lb.to_string(index=False))

    best_name = lb.iloc[0]["model"]
    best_thr  = float(lb.iloc[0]["val_thr"])
    print(f"\n>>> WINNER: {best_name}   (decision threshold = {best_thr:.2f})")

    # --- 3. refit winner on TRAIN+VAL, evaluate on untouched TEST ---
    best_pipe = build_model_zoo(pos_weight)[best_name]
    best_pipe.fit(X_tr, y_tr)
    p_te  = best_pipe.predict_proba(X_te)[:, 1]
    pred  = (p_te >= best_thr).astype(int)
    test_metrics = score_at(y_te, p_te, best_thr)

    print("\n--- TEST-SET PERFORMANCE (untouched data) ---")
    for k, v in test_metrics.items():
        print(f"   {k:12s}: {v}")
    print("\nClassification report (1 = rare/negative case):")
    print(classification_report(y_te, pred, target_names=["0_benign","1_RARE"], zero_division=0))
    cm = confusion_matrix(y_te, pred)
    print("Confusion matrix [rows=true, cols=pred]:\n", cm)

    # --- 4. permutation importance (which combinations matter) ---
    top_imp = None
    try:
        cap = min(CONFIG["perm_importance_sample"], len(X_te))
        idx = np.random.RandomState(RS).choice(len(X_te), cap, replace=False)
        pi = permutation_importance(best_pipe, X_te.iloc[idx], y_te[idx],
                                    scoring="roc_auc", n_repeats=CONFIG["perm_importance_repeats"],
                                    random_state=RS, n_jobs=-1)
        top_imp = (pd.Series(pi.importances_mean, index=FEATURE_COLS)
                   .sort_values(ascending=False).head(10))
        print("\nTop-10 features (permutation importance on ROC-AUC):")
        print(top_imp.to_string())
    except Exception as e:
        print("   [permutation importance skipped]:", e)

    # --- 5. plots ---
    if show_plots:
        ncols = 2 if top_imp is not None else 1
        fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 4))
        axes = np.atleast_1d(axes)
        axes[0].imshow(cm, cmap="Blues")
        axes[0].set_title(f"{short}: confusion matrix")
        axes[0].set_xticks([0,1]); axes[0].set_xticklabels(["pred 0","pred 1"])
        axes[0].set_yticks([0,1]); axes[0].set_yticklabels(["true 0","true 1"])
        for (i,j), v in np.ndenumerate(cm):
            axes[0].text(j, i, str(v), ha="center", va="center",
                         color="white" if v > cm.max()/2 else "black")
        if top_imp is not None:
            top_imp[::-1].plot.barh(ax=axes[1])
            axes[1].set_title(f"{short}: top feature importance")
        plt.tight_layout(); plt.show()

    # --- 6. save artifact ---
    artifact = {"pipeline": best_pipe, "threshold": best_thr, "model_name": best_name,
                "question": qcol, "short_name": short, "class1_meaning": meaning,
                "feature_cols": FEATURE_COLS, "test_metrics": test_metrics}
    path = os.path.join(CONFIG["output_dir"], f"model_{short}.joblib")
    joblib.dump(artifact, path)
    print(f"\n[saved] -> {path}")

    return {"question": short, "model": best_name, "threshold": round(best_thr,2),
            "rare_rate": round(float(y_all.mean()),4), **test_metrics}

RESULTS = {}   # collects one row per question for the final summary

## `BOOKMARK 10` — Train one model per question

Each cell below trains, evaluates and saves the model for a single question. They are independent —
run them in any order (after sections 1–9). Re-run a single cell to retrain just that question.

### `BOOKMARK 10.1` — Q1: Jealousy (husband jealous if wife talks with other men)

In [ ]:
# ===== BOOKMARK 10.1 :: Q1 Jealousy =====
RESULTS["Husband_partner_jealous_if_Wife_talks_with_other_men"] = train_question(
    "Husband_partner_jealous_if_Wife_talks_with_other_men")

### `BOOKMARK 10.2` — Q2: Controlling (insists on knowing where wife is)

In [ ]:
# ===== BOOKMARK 10.2 :: Q2 Controlling =====
RESULTS["Husband_partner_insists_on_knowing_where_Wife_is"] = train_question(
    "Husband_partner_insists_on_knowing_where_Wife_is")

### `BOOKMARK 10.3` — Q3: Humiliated

In [ ]:
# ===== BOOKMARK 10.3 :: Q3 Humiliated =====
RESULTS["Ever_been_humiliated_by_husband_partner"] = train_question(
    "Ever_been_humiliated_by_husband_partner")

### `BOOKMARK 10.4` — Q4: Insulted / made to feel bad

In [ ]:
# ===== BOOKMARK 10.4 :: Q4 Insulted =====
RESULTS["Ever_been_insulted_or_made_to_feel_bad_by_husband_partner"] = train_question(
    "Ever_been_insulted_or_made_to_feel_bad_by_husband_partner")

### `BOOKMARK 10.5` — Q5: Pushed / shook / had something thrown

In [ ]:
# ===== BOOKMARK 10.5 :: Q5 Pushed-Shook =====
RESULTS["Ever_been_pushed,_shook_or_had_something_thrown_by_husband_partner"] = train_question(
    "Ever_been_pushed,_shook_or_had_something_thrown_by_husband_partner")

### `BOOKMARK 10.6` — Q6: Slapped

In [ ]:
# ===== BOOKMARK 10.6 :: Q6 Slapped =====
RESULTS["Ever_been_slapped_by_husband_partner"] = train_question(
    "Ever_been_slapped_by_husband_partner")

### `BOOKMARK 10.7` — Q7: Health-care decision agency

In [ ]:
# ===== BOOKMARK 10.7 :: Q7 Health decision =====
RESULTS["Person_who_usually_decides_on_Wife's_health_care"] = train_question(
    "Person_who_usually_decides_on_Wife's_health_care")

### `BOOKMARK 10.8` — Q8: Large-purchase decision agency

In [ ]:
# ===== BOOKMARK 10.8 :: Q8 Purchase decision =====
RESULTS["Person_who_usually_decides_on_large_household_purchases"] = train_question(
    "Person_who_usually_decides_on_large_household_purchases")

### `BOOKMARK 10.9` — Q9: Fear (wife afraid most of the time)

In [ ]:
# ===== BOOKMARK 10.9 :: Q9 Fear =====
RESULTS["Wife_afraid_of_husband_partner_most_of_the_time,_sometimes_or_never"] = train_question(
    "Wife_afraid_of_husband_partner_most_of_the_time,_sometimes_or_never")

## `BOOKMARK 11` — Summary across all 9 questions

In [ ]:
# ===== BOOKMARK 11 :: SUMMARY =====
summary = pd.DataFrame(RESULTS.values())
cols = ["question","rare_rate","model","threshold","MCC","BalancedAcc",
        "Recall_rare","Prec_rare","F1_rare","PR_AUC","ROC_AUC"]
summary = summary[cols].sort_values("MCC", ascending=False).reset_index(drop=True)
print("PER-QUESTION RESULTS (test set, 1 = rare/negative case)")
try:
    display(summary)
except NameError:
    print(summary.to_string())
summary.to_csv(os.path.join(CONFIG["output_dir"], "summary_metrics.csv"), index=False)
print("\nReading the table:")
print(" * Recall_rare  = % of true rare cases the model catches (your main goal).")
print(" * Prec_rare    = of the cases flagged rare, % that really are.")
print(" * MCC / BalancedAcc near 0 => weak signal; near 1 => strong, balanced.")

## `BOOKMARK 12` — Load a saved model & predict on new couples

In [ ]:
# ===== BOOKMARK 12 :: INFERENCE EXAMPLE =====
def predict_question(short_name, new_df):
    """Load a saved model and return (label, probability) for new rows.
       new_df must contain the 13 RAW_FEATURES columns; engineering is re-applied."""
    art = joblib.load(os.path.join(CONFIG["output_dir"], f"model_{short_name}.joblib"))
    X = engineer_features(new_df)[art["feature_cols"]]
    proba = art["pipeline"].predict_proba(X)[:, 1]
    label = (proba >= art["threshold"]).astype(int)
    return label, proba

# Example: score the first 5 couples with the 'Slapped' model
labels, probs = predict_question("Slapped", df.head(5))
print("predicted rare-class label:", labels)
print("predicted probability     :", np.round(probs, 3))

---
### Notes on results & honesty
These outcomes are predicted **from demographics only**, which carry a genuinely *weak* signal for
intimate-partner behaviour. Expect **good recall on the rare class** (the notebook is tuned for that)
but **modest precision / MCC** — that is a real property of the data, not a bug. To shift the
recall⇄precision balance, change `CONFIG["threshold_objective"]` (e.g. `"f2"` for more recall) and
re-run. If you install `xgboost`, `lightgbm` and `imbalanced-learn`, they are added to the zoo
automatically and usually improve the boosting candidates.